# Notebook 04 — Scenario Modifier Design

This notebook justifies and calibrates the multiplicative modifier table used in `forecaster.py`.

Each scenario shifts the Prophet trend by a multiplier on the projected change from 2025 baseline.
The magnitudes are loosely grounded in IEA WEO 2023 and IPCC AR6 ranges.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt, numpy as np
from apps.api.services.forecaster import SCENARIO_MODIFIERS, run_forecast

# Generate a synthetic linear history for illustration
def synthetic_history(start, end, years=range(1985, 2026), null_pre=2000):
    return [{'year': y, 'value': start + (end - start) * (y - 1985) / 40 if y >= null_pre else None, 'source': 'synthetic'}
            for y in years]

mock = {
    'ndvi': synthetic_history(0.55, 0.38),
    'nighttime_lights': synthetic_history(3.0, 38.0),
    'land_surface_temp': synthetic_history(27.0, 29.0, null_pre=2000),
    'built_up_extent': synthetic_history(50, 420),
}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
signals = ['ndvi', 'nighttime_lights', 'land_surface_temp', 'built_up_extent']
colors = {'bau': '#94a3b8', 'solar': '#fbbf24', 'grid': '#4f8ef7', 'stress': '#f87171'}

for ax, sig in zip(axes.flat, signals):
    # Plot history
    hist_vals = [pt for pt in mock[sig] if pt['value'] is not None]
    ax.plot([p['year'] for p in hist_vals], [p['value'] for p in hist_vals],
            'o-', color='white', lw=1.5, ms=2, label='Historical', zorder=5)
    ax.axvline(2025, color='#4f8ef7', linestyle='--', alpha=0.5)

    for scenario, color in colors.items():
        result = run_forecast(mock, scenario)
        fore = result['signals'][sig]
        years = [p['year'] for p in fore]
        vals = [p['value'] for p in fore]
        lowers = [p.get('lower', p['value']) for p in fore]
        uppers = [p.get('upper', p['value']) for p in fore]
        ax.plot(years, vals, '-', color=color, lw=1.5, label=scenario)
        ax.fill_between(years, lowers, uppers, color=color, alpha=0.08)

    ax.set_title(sig.replace('_', ' ').title())
    ax.legend(fontsize=8)
    ax.set_facecolor('#0f1117')

fig.patch.set_facecolor('#0f1117')
plt.suptitle('Scenario divergence — synthetic Bangalore-like coordinate', color='white')
plt.tight_layout()
plt.show()

## Modifier rationale

| Signal | Solar modifier | Rationale |
|--------|---------------|----------|
| NDVI × 1.1 | Solar replaces degraded land with panels that harbour sparse vegetation, net green |
| LST × 0.85 | Reduced fossil combustion + albedo change from panels reduces urban heat |
| Lights × 0.9 | Grid efficiency → same activity, fewer watt-hours → lower VIIRS radiance |

| Signal | Climate Stress modifier | Rationale |
|--------|------------------------|----------|
| NDVI × 0.8 | Drought stress + heat → vegetation die-back |
| LST × 1.25 | Compound warming |

All modifiers should be revisited against actual policy data for a v2 physical model.